# 05-03. 전체 통계·시간 변화·공통 패턴

**담당:** 정수진 · **Issue:** #65

02에서 본 실제 파형과 비교 위치를 수치로 확인한다. 14 LOT의 수준·변동·시간 변화와 여러 LOT에서 반복되는 구역별 패턴을 모아 읽는다.

| 파일 | 읽는 흐름 |
|---|---|
| [01 자료·품질·시간 지도](05_all_data_timeseries_jinny_01.ipynb) | 자료 범위 → 연간 흐름 → 월별 관측창·정비 → 품질 |
| [02 LOT·강종별 연결 대조](05_all_data_timeseries_jinny_02.ipynb) | 각 LOT 전체 → 특이 구간 확대 → 이력 → 동일 강종 비교 → 해당 원본 |
| [03 전체 통계·공통 패턴](05_all_data_timeseries_jinny_03.ipynb) | 수준·IQR·시간 변화 → 구역별 응답 → CP·연관성 |
| [04 탐지·구간 근거](05_all_data_timeseries_jinny_04.ipynb) | 기준선·이동 Z·IsolationForest → 후보 위치 → 근거·해석 범위 |

대상은 CAL01의 제공된 14 LOT·22개 센서와 CAL01 정비/같은 WO의 부품교체, CR1 강종변경·정기수리다. 관측 범위는 실제 가동시간이나 소재 입·출측 시간이 아니다. 같은 강종·같은 경과 분은 비교 기준이며 같은 설정값·속도·두께를 보장하지 않는다. CP-Z4·CP-Z4M은 무차원이며, 물리적 정의와 산식은 별도로 확인해야 한다. 시간 겹침·선후관계만으로 고장 원인이나 정비 효과를 확정하지 않는다.

**이 파일에서 바로 보기:** [수준과 흐름](#level-variation) · [출력·온도 응답](#response-patterns) · [과시효·세정](#process-levels) · [CP와 동시 신호](#cp-patterns) · [온도·출력 연관성](#relationships) · [전수 원본 근거](#all-raw-evidence)

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 무차원 기록값으로 원본 값을 그대로 표시합니다.

In [ ]:
from pathlib import Path
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/T-CR1-CAL01_온도.csv").exists() and (p / "notebooks").is_dir())
DATA_DIR = str(PROJECT_ROOT / "data")
FIG_DIR = str(PROJECT_ROOT / "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

기존 01과 같은 규칙으로 읽습니다. LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "03"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":"#b52a49","CBM":"#007f86","TBM":"#825a30"})

## 그림 표시 규칙

기존 01~04의 그림 규칙을 그대로 씁니다. 짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용하고 TC는 왼쪽·대응 OP는 오른쪽, CP-Z4는 왼쪽·CP-Z4M은 오른쪽에 둡니다. 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

**추가한 규칙:** 모든 그림 오른쪽 위에 시간축 태그를 붙입니다. 실제 사건·원본 확대는 `TIME AXIS · 실제 측정 날짜·시각`, 여러 LOT 형태 비교는 `TIME AXIS · 관측 시작 후 경과시간[분]`, 시간축이 없는 요약·결과표는 `COMPARISON · 비교 대상`입니다. 실제 시각 원본 확대에는 위쪽 보조축으로 같은 LOT의 경과 분을 함께 적어, 경과시간 비교 그림의 위치를 바로 찾을 수 있게 합니다.

**CP 표기:** CP-Z4와 CP-Z4M은 모두 무차원이다. 그래프 축은 각각 `CP-Z4 기록값`, `CP-Z4M 기록값`으로 표시한다. CP-Z4M의 `1e-10` 또는 `×1e-10`은 눈금 배율이며 단위가 아니다. 계측 산식과 설비·부품 연결은 단위와 별도로 확인한다.

**범례 이름:** 색·선·마커가 뜻하는 자료와 작업을 적는다. 작업 유형·내용·시간은 남기고, 띠 순서나 화면 위치는 이름에 붙이지 않는다. 확대 표시 `(a)~(c)`와 실제 시간의 전·후 구분은 유지한다.

노트북에는 코드가 그린 그림이 바로 표시되고, 같은 그림이 원본 해상도 PNG로 `figures/05_all_data_timeseries_jinny_03/<구분 폴더>/`에 저장된다. 계산 코드는 접어 두고 그림·표·해석은 펼쳐 두었다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값"
    if signal == "CP-Z4":
        return "CP-Z4 기록값"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.8인치) 둔다.
PAD = .8
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    layout = compact_explanations(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(160dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display(fig)
    plt.close(fig)

In [ ]:
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

## 상세 시계열 준비

1초 기대 간격의 품질, 1분 집계, 사후 형태 상태(초기 승온·종료 전환·안정 참고), 같은 강종·안정 참고·같은 경과 분 비교, 실제 작업 교집합을 계산한다. 기존 01~08과 같은 코드다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8)
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 무차원이다. 자체 참고 중앙값과 차이를 탐색하며 정상 한계로 해석하지 않는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

In [ ]:
hypotheses = []
def verdict(hypothesis, result, evidence, reason, use, limit="운전 설정·인과·실제 고장 정보 부족", source="기존 03"):
    hypotheses.append({"가설": hypothesis, "판정": result, "검증 근거": evidence, "판정 이유": reason,
                       "08 활용": use, "남은 한계": limit, "출처": source})

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def clock_axis(ax, day):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    style(ax, f"실제 측정 시각[{day:%m/%d} 시:분]")

def clock_top(ax, start, label="같은 LOT 실제 시각[시:분]"):
    """경과 분 축 위에 대상 LOT의 실제 시각을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda m: base + m / 1440, lambda d: (d - base) * 1440))
    top.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    return top

def mean30(k, cols):
    """팀 자료와 같은 집계: 센서별 유효값 30개 이상인 1분 평균."""
    g = raw[k].set_index("MEAS_DT")[cols]
    return g.resample("1min").mean().where(g.resample("1min").count().ge(30))

def sustained_from(series, limit):
    """기준보다 낮은 상태가 관측 끝까지 이어지는 구간의 시작 시각."""
    s = series.dropna(); low = s.lt(limit)
    if not low.iloc[-1]:
        return None
    i = len(low) - 1
    while i > 0 and low.iloc[i - 1]:
        i -= 1
    return low.index[i]

In [ ]:
# ------------------------------------------------------------------------------------------------
# 예정된 변화와 확실히 볼 후보 ①~④ (01·02·03 노트북이 같은 코드로 계산한다)
# ------------------------------------------------------------------------------------------------
# 예정된 변화: 계획수리 직전의 종료 전환(끄면 출력·온도가 함께 내려감), 직후의 초기 승온(켜면 함께 올라감)
relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "예정된 정지"), ("초기 승온", "재가동 승온")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())

# ① 출력과 온도가 반대로 움직인 구간: 출력 0%(≤2%)가 5분 이상인데 온도가 내려가지 않음 / 출력 100%(≥98%)가 5분 이상인데 온도가 오르지 않음
opposite_rows = []
for k in lots.LOT_NO.astype(int):
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9))
        tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on, bad in [("출력 0%인데 온도 안 내려감", op.le(2), lambda d: d >= 0),
                              ("출력 100%인데 온도 안 오름", op.ge(98), lambda d: d <= 0)]:
            for a, b in runs(on.fillna(False), 5):
                change = tc.iloc[min(b, len(tc) - 1)] - tc.iloc[a]
                if bad(change):
                    opposite_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "ZONE": f"Z{z}", "종류": kind,
                                          "START_MIN": a, "END_MIN": b, "길이[분]": b - a, "온도 변화[℃]": change,
                                          "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=a),
                                          "예정된 변화 중": in_expected(k, a)})
opposite = pd.DataFrame(opposite_rows)

# ②·③ 같은 강종 대비 LOT 전체 수준 차이: 안정 참고 1분 중앙값의 LOT 중앙값 - 같은 강종 다른 LOT 중앙값들의 중앙값
level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(states[k].eq("안정 참고") & quality[k][s].ge(.9)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN
overage = level_gap.loc[level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.ge(2.5)]       # ② 과시효 Z7·Z8이 같은 강종보다 2.5℃ 이상 높은 LOT
OVERAGE_LOTS = sorted(overage.LOT.unique())

# ③ 세정 입구 온도(CLN-IN) LOT 중앙값을 정렬해 가장 큰 간격에서 두 묶음으로 나눈다.
cln = level_gap.loc[level_gap.SENSOR.eq("CLN-IN")].set_index("LOT").LOT_MEDIAN.sort_values()
cut = cln.diff().idxmax()                                   # 큰 묶음의 첫 LOT
CLN_SPLIT = (cln.loc[:cut].iloc[-2] + cln.loc[cut]) / 2
cln_group = pd.Series(np.where(cln.ge(CLN_SPLIT), f"{CLN_SPLIT:.1f}℃ 이상", f"{CLN_SPLIT:.1f}℃ 미만"), index=cln.index)

# ④ CP 변화: 자체 안정 참고 기준과 가장 크게 벗어난 펄스 (예정된 정지 중의 CP 하강은 제외)
cp_main = cp_candidates.loc[[not in_expected(int(r.LOT), r.PEAK_MIN) for r in cp_candidates.itertuples()]]
CP_TOP = cp_main.loc[cp_main.DELTA.abs().idxmax()]

# ① 대표 사례: 출력 0%가 5분 이상인데 온도가 가장 크게 오른 곳
z0 = opposite.loc[opposite["종류"].str.startswith("출력 0%") & ~opposite["예정된 변화 중"]]
Z3_TOP = z0.loc[z0["온도 변화[℃]"].idxmax()]

# 03에서 자세히 본 LOT과 이유 (그림에서 빨간색으로 표시한다)
SELECTED = {}
for k in OVERAGE_LOTS:
    SELECTED.setdefault(int(k), []).append("② Z7·Z8")
SELECTED.setdefault(int(CP_TOP.LOT), []).append("④ CP")
SELECTED.setdefault(int(Z3_TOP.LOT), []).append("① Z3")
SELECT_COLOR = "#d62728"

MARK = {"①": "#c0392b", "②": "#8e44ad", "③": "#16a085", "④": "#d35400"}
MARK_NAME = {"①": "출력·온도 반대 움직임", "②": "과시효 Z7·Z8 수준 차이", "③": "세정 입구 온도 두 수준", "④": "가장 큰 CP 변화"}
EXPECTED_COLOR = "#8d99a6"

In [ ]:
from matplotlib.patches import Rectangle
GRADE_COLOR={"SGCC":"#455a64","SPFH590":"#8c564b","SPCE":"#0072b2","SPCEN":"#cc79a7","SPHC":"#009e73","DP590":"#7f7f00","SPCD":"#7b2cbf","SPCC":"#8c6d31"}
ZONE_COLOR={1:"#455a64",2:"#d55e00",3:"#0072b2",4:"#e69f00",5:"#009e73",6:"#cc79a7",7:"#8c6d31",8:"#7b2cbf",9:"#56b4e9"}
ZONES=[3,4,5,6]
SELECT_COLOR="#c83745"
MARK_NAME["①"]="출력·온도 응답 확인"
PROCESS["TC-Z9"]="Z9"
def sensor_color(s):
    if s.startswith(("TC-Z","OP-Z")):return ZONE_COLOR[int(s.rsplit('Z',1)[1])]
    return {"CLN-IN":"#455a64","CP-Z4":"#b04482","CP-Z4M":"#7b2cbf","TC-OUT1":"#009e73","TC-OUT2":"#d55e00"}[s]
def sensor_scale(s):return 1e10 if s=="CP-Z4M" else 1
def shown_unit(s):return "CP-Z4M 기록값[×1e-10]" if s=="CP-Z4M" else unit(s)
def when(k,a,b):
    start=lot_meta.loc[k,"START"]
    return f"{start+pd.Timedelta(minutes=float(a)):%m/%d %H:%M:%S}–{start+pd.Timedelta(minutes=float(b)):%m/%d %H:%M:%S}"
def clock_axis(ax,day=None):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    style(ax,"실제 측정 날짜·시각[월/일 시:분]")
def raw_window(lot,a,b):
    r=lot_meta.loc[lot]
    times=pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)),min(r.END,r.START+pd.Timedelta(seconds=int(b*60))),freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)
def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"
def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]
def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 모든 후보를 목록에 남기고, 긴 수준 차이와 짧은 변화를 구분해 표시한다.
review_rows=[]
for e in episodes.itertuples():
    level=level_gap.loc[level_gap.LOT.eq(e.LOT)&level_gap.SENSOR.eq(e.SENSOR),"GAP"].iloc[0]
    long=bool(e.DURATION_MIN>=30)
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":e.SENSOR,"KIND":"같은 강종 차이","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"최대 {e.MAX_ABS_DELTA_C:.2f}℃ · 대조 {e.MIN_PEERS} LOT 이상","LOCAL":not long})
for e in opposite.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":f"TC-{e.ZONE}","KIND":"출력·온도 응답","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"{e.종류} · TC 변화 {e._8:+.2f}℃","LOCAL":True})
for e in cp_candidates.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":"CP-Z4","KIND":"CP 변화","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"자체 참고 대비 {e.DELTA:+.3f} · 극값 {e.PEAK_VALUE:.4g}","LOCAL":e.PULSE_WIDTH_MIN<30})
review_catalogue=pd.DataFrame(review_rows).sort_values(["LOT","A","SENSOR"]).reset_index(drop=True)
review_catalogue["REVIEW_ID"]=[f"R{i+1:03d}" for i in range(len(review_catalogue))]
review_catalogue["START"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.A,unit="min")
review_catalogue["END"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.B,unit="min")

def lot_windows(k):
    """겹친 국소 변화의 문맥을 합친다. 긴 수준 차이는 전체 파형·비교 표에서 확인한다."""
    rows=review_catalogue.loc[review_catalogue.LOT.eq(k)&review_catalogue.LOCAL]
    result=[]
    for r in rows.sort_values('A').itertuples():
        a=max(0,r.A-5);b=min(lot_meta.loc[k,"MINUTES"],r.B+5)
        if result and a<=result[-1]["B"]+3:
            result[-1]["B"]=max(result[-1]["B"],b);result[-1]["IDS"].append(r.REVIEW_ID);result[-1]["SENSORS"].add(r.SENSOR)
        else:result.append({"A":a,"B":b,"IDS":[r.REVIEW_ID],"SENSORS":{r.SENSOR}})
    for i,r in enumerate(result):r["WINDOW_ID"]=f"{k}-W{i+1:02d}"
    return result
overview_coverage={};window_coverage={};maintenance_stage_rows=[];window_audit=[]

<a id="level-variation"></a>

## 1. 안정 참고 구간의 수준과 흐름

In [ ]:
# 강종 색은 온도 선(주황)·볼 LOT 표시(빨강)와 겹치지 않게 고른다
GRADE_COLOR={"SGCC":"#455a64","SPFH590":"#8c564b","SPCE":"#0072b2","SPCEN":"#cc79a7","SPHC":"#009e73","DP590":"#7f7f00","SPCD":"#7b2cbf"}
state_stats=[]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for state in states[k].unique():
        for s in SIGNALS:
            y=profile[k][(s,"median")].where(states[k].eq(state)&quality[k][s].ge(.9)).dropna()
            state_stats.append({"LOT":k,"GRADE":r.GRADE,"STATE":state,"SENSOR":s,"N_WINDOWS":len(y),"MEDIAN":y.median(),
                                "IQR":y.quantile(.75)-y.quantile(.25),
                                "FIRST30":y.head(30).median() if len(y)>=60 else np.nan,
                                "LAST30":y.tail(30).median() if len(y)>=60 else np.nan,
                                "FIRST_LAST30_DELTA":y.tail(30).median()-y.head(30).median() if len(y)>=60 else np.nan,
                                "SLOPE_PER_MIN":np.polyfit(y.index.to_numpy(),y.to_numpy(),1)[0] if len(y)>=20 else np.nan})
state_stats=pd.DataFrame(state_stats)

# 같은 강종을 붙여 놓고, 강종 사이에는 빈 줄을 둔다.
order=lots.assign(G=lots.GRADE.map(list(GRADE_COLOR).index)).sort_values(["G","LOT_NO"])
ypos,y,prev={},0,None
for r in order.itertuples():
    y+=.7 if prev not in [None,r.GRADE] else 0
    ypos[int(r.LOT_NO)]=y;y+=1;prev=r.GRADE

def stable_values(k,s):
    return profile[k][(s,"median")].where(states[k].eq("안정 참고")&quality[k][s].ge(.9)).dropna()

In [ ]:
# LOT별 Z3 수준: 점 = 안정 참고 구간 중앙값, 막대 = 가운데 50% (같은 강종끼리 묶음)
by_grade = order.LOT_NO.astype(int).tolist()
pos, x, prev = {}, 0, None
for k in by_grade:
    g = lot_meta.loc[k, "GRADE"]; x += .8 if prev not in [None, g] else 0; pos[k] = x; x += 1; prev = g
fig, axs = plt.subplots(1, 2, figsize=(30, 11)); fig.subplots_adjust(left=.05, right=.98, top=.75, bottom=.2, wspace=.15)
level_med = {}
for ax, (s, u) in zip(axs, [("TC-Z3", "℃"), ("OP-Z3", "%")]):
    for k in by_grade:
        v = stable_values(k, s); c = GRADE_COLOR[lot_meta.loc[k, "GRADE"]]; level_med[k, s] = v.median()
        ax.plot([pos[k], pos[k]], [v.quantile(.25), v.quantile(.75)], color=c, lw=14, alpha=.35, solid_capstyle="butt")
        ax.plot(pos[k], v.median(), "o", color=c, ms=12, zorder=3)
        ax.text(pos[k], v.quantile(.75), f"{v.median():.1f}", ha="center", va="bottom", fontsize=12, color=c)
    ax.set_xticks([pos[k] for k in by_grade], [str(k) for k in by_grade], rotation=35, ha="right")
    for t, k in zip(ax.get_xticklabels(), by_grade):
        t.set_color(GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
    for g in dict.fromkeys(lot_meta.loc[by_grade, "GRADE"]):
        xs_g = [pos[k] for k in by_grade if lot_meta.loc[k, "GRADE"] == g]
        ax.text(np.mean(xs_g), -.24, g, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=13, fontweight="bold", color=GRADE_COLOR[g])
    style(ax, ""); ax.set_ylabel(f"{'온도' if s == 'TC-Z3' else '제어출력'} {s} [{u}]", fontsize=13); ax.tick_params(labelsize=12)
    vals = pd.Series({k: level_med[k, s] for k in by_grade})
    if s == "TC-Z3":
        ax.set_ylim(857, 863)
        ax.set_title(f"(a) 온도: 14 LOT 중앙값이 {vals.min():.1f}~{vals.max():.1f}℃ — 차이 {vals.max() - vals.min():.1f}℃로 거의 같다", loc="left", fontsize=17, pad=12)
    else:
        sphc = vals[[k for k in by_grade if lot_meta.loc[k, "GRADE"] == "SPHC"]]; rest = vals.drop(sphc.index)
        ax.set_ylim(50, 100)
        ax.set_title(f"(b) 제어출력: SPHC는 {sphc.min():.0f}~{sphc.max():.0f}%, 나머지 강종은 {rest.min():.0f}~{rest.max():.0f}% — 강종마다 다르다", loc="left", fontsize=17, pad=12)
header(fig, "안정 참고 구간 · 가열·균열 Z3 수준", "점 = LOT 중앙값 · 굵은 막대 = 가운데 50% · 같은 강종끼리 묶음(아래 굵은 글씨 = 강종)", [], ncol=1)
compact_header(fig, .89, .86, title_size=26, sub_size=14)
axis_tag(fig, compare_tag("14 LOT 가열·균열 Z3"))
finish(fig, "01_수준변동/01_안정참고_Z3수준.png")
tc_v = pd.Series({k: level_med[k, "TC-Z3"] for k in by_grade}); op_v = pd.Series({k: level_med[k, "OP-Z3"] for k in by_grade})
note("\n".join([
    f"- (a) 온도 TC-Z3의 LOT 중앙값은 {tc_v.min():.1f}~{tc_v.max():.1f}℃로, 강종이 달라도 {tc_v.max() - tc_v.min():.1f}℃ 안에 있다. 온도는 강종과 관계없이 거의 같은 값으로 운전된다.",
    f"- (b) 같은 구역의 제어출력 OP-Z3는 {op_v.min():.0f}~{op_v.max():.0f}%로 넓게 퍼진다. 온도를 같게 맞추기 위해 강종마다 다른 출력을 쓴다."]))
table(state_stats)


In [ ]:
# LOT마다 한 칸: 왼쪽 축 = 온도 TC-Z3, 오른쪽 축 = 제어출력 OP-Z3. 둘 다 1분 값으로 그려 몇 분짜리 오르내림까지 같은 시점에서 본다.
date_order = lots.LOT_NO.astype(int).tolist()
OP_COLOR, TC_COLOR = "#111111", "#d55e00"   # 온도 = 주황, 출력 = 검정
series = {k: (stable_values(k, "OP-Z3"), stable_values(k, "TC-Z3")) for k in date_order}
# 두 축 모두 LOT 중앙값을 가운데에 두고, 높이 1℃ = 출력 10%p로 맞춘다(선의 위아래 위치가 아니라 움직인 방향만 비교하도록).
# 10은 14 LOT에서 1분 값의 출력이 보통 움직이는 폭이 온도 폭의 약 9배라서다(1분 값의 10~90% 폭, LOT별 출력/온도 비 중앙값 9.3).
OP_PER_C = 10
tc_half = max(4, np.ceil(max((t - stable_values(k, "TC-Z3").median()).abs().max() for k, (o, t) in series.items())),
              np.ceil(max((o - stable_values(k, "OP-Z3").median()).abs().max() for k, (o, t) in series.items()) / OP_PER_C))
fig, axs = plt.subplots(7, 2, figsize=(30, 30)); fig.subplots_adjust(left=.05, right=.95, top=.875, bottom=.04, hspace=.62, wspace=.2)
flow_rows = []
for ax, k in zip(axs.T.ravel(), date_order):
    o, t = series[k]; m0 = stable_values(k, "TC-Z3").index.min()
    tm = stable_values(k, "TC-Z3").median()
    bx = ax.twinx()
    ax.axhspan(tm - 1, tm + 1, color="#e8edf1", lw=0, zorder=0)
    ax.plot(t.index - m0, t, color=TC_COLOR, lw=1.3)
    bx.plot(o.index - m0, o, color=OP_COLOR, lw=1.1, alpha=.85)
    om = stable_values(k, "OP-Z3").median(); oh = tc_half * OP_PER_C
    ax.axhline(tm, color="#9fb3c3", lw=.8, ls=":")
    ax.set_ylim(tm - tc_half, tm + tc_half); bx.set_ylim(om - oh, om + oh)
    bx.set_yticks([v for v in range(0, 101, 25) if om - oh <= v <= om + oh])
    n = len(t); both = pd.concat([o, t], axis=1, keys=["OP", "TC"]).dropna()
    rho = both.OP.corr(both.TC, method="spearman")
    end_t = (t.iloc[int(n * .9):] - tm).median(); end_o = (o.iloc[int(n * .9):] - stable_values(k, "OP-Z3").median()).median()
    together = "반대로 움직임" if rho <= -.4 else "같이 움직임" if rho >= .4 else "관계 뚜렷하지 않음"
    big = abs(end_t) >= 2
    ax.set_title(f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d} · 안정 {n}분\n"
                 f"{together} ({rho:+.2f}) · 마지막 10%: 온도 {end_t:+.1f}℃ / 출력 {end_o:+.1f}%p",
                 loc="left", fontsize=14, pad=8, color=GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
    if big:   # 볼 LOT: 제목 앞에 빨간 ★
        ax.annotate("★", (0, 1), xycoords="axes fraction", xytext=(-8, 25), textcoords="offset points", ha="right", va="baseline", fontsize=20, color=SELECT_COLOR)
    style(ax, ""); ax.tick_params(labelsize=11); ax.tick_params(axis="y", colors=TC_COLOR); bx.tick_params(labelsize=11, colors=OP_COLOR, labelbottom=False)   # 출력 축은 가로 눈금 숫자를 쓰지 않는다
    bx.spines[["top"]].set_visible(False); bx.spines["right"].set_color("#a9bac7")
    ax.set_ylabel("온도[℃]", fontsize=11.5, color=TC_COLOR); bx.set_ylabel("제어출력[%]", fontsize=11.5, color=OP_COLOR)
    ax.set_xlim(0, max(n, 60))
    for s, y_, e_ in [("TC-Z3", stable_values(k, "TC-Z3"), end_t), ("OP-Z3", stable_values(k, "OP-Z3"), end_o)]:
        flow_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "SENSOR": s, "안정 구간[분]": len(y_),
                          "기울기[100분당]": np.polyfit(np.arange(len(y_)), y_.to_numpy(), 1)[0] * 100, "마지막 10% 값": e_, "온도·출력 관계(순위상관)": rho})
for ax in axs[-1]:
    ax.set_xlabel("안정 참고 구간 시작 후[분]", fontsize=12)
flow_table = pd.DataFrame(flow_rows)
header(fig, "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기",
       "LOT마다 한 칸 · 왼쪽 주황 = 온도 TC-Z3 · 오른쪽 검정 = 제어출력 OP-Z3 (둘 다 1분 값) · 두 축 모두 LOT 중앙값을 가운데(점선)에, 높이 1℃ = 10%p · 회색 띠 = ±1℃(= ±10%p) · 칸 제목 색 = 강종",
       [Line2D([0], [0], color=TC_COLOR, lw=2.2, label="온도 TC-Z3 (왼쪽 축)"), Line2D([0], [0], color=OP_COLOR, lw=1.8, label="제어출력 OP-Z3 (오른쪽 축)"),
        Patch(facecolor="#e8edf1", label="중앙값 ±1℃ (= 출력 ±10%p)"),
        Line2D([0], [0], marker="*", ls="None", color=SELECT_COLOR, ms=18, label="볼 LOT (마지막 10% 온도가 평소보다 2℃ 넘게 다름)")], ncol=4)
compact_header(fig, .955, .935, title_size=26, sub_size=14, legend_size=14)
# 둘째 줄: 칸 제목 색(강종)을 한 줄로 묶어 적는다
fig.legend(handles=[Line2D([], [], ls="None", label="칸 제목 색 = 강종:")]
           + [Line2D([0], [0], marker="s", ls="None", color=GRADE_COLOR[g], ms=12, label=g) for g in GRADE_COLOR if g in set(lot_meta.loc[date_order, "GRADE"])],
           loc="upper left", bbox_to_anchor=(.031, .918), ncol=8, frameon=False, fontsize=14, handlelength=1.2, columnspacing=1.8)
axis_tag(fig, "TIME AXIS · 안정 참고 구간 시작 후[분]")
fig.text(.035, .006, "칸 제목 읽는 법 · 마지막 10% = LOT 기록 끝 10% 시간의 중앙값이 LOT 전체 중앙값과 다른 정도 · ★ = 그 온도 차이가 2℃ 넘는 LOT · 제목 색 = 강종",
         ha="left", va="bottom", fontsize=12.5, color="#526d80")
finish(fig, "01_수준변동/02_안정참고_Z3출력과온도.png")
ft = flow_table.loc[flow_table.SENSOR.eq("TC-Z3")].set_index("LOT"); fo = flow_table.loc[flow_table.SENSOR.eq("OP-Z3")].set_index("LOT")
opp = ft.loc[ft["온도·출력 관계(순위상관)"].le(-.4), "온도·출력 관계(순위상관)"].sort_values()
endbig = ft.index[ft["마지막 10% 값"].abs().ge(2)].tolist()
note("\n".join([
    "- 같은 칸에서 검정(출력)과 주황(온도)이 **반대로** 움직이면, 온도가 내려갈 때 제어가 출력을 올려 보충한 것이다. 출력은 그대로인데 온도만 움직이면 출력 말고 다른 이유를 찾아야 한다.",
    f"- 반대로 움직인 정도가 뚜렷한 LOT(순위상관 -0.4 이하)은 " + ", ".join(f"{k}({r:+.2f})" for k, r in opp.items()) + f"이다 ({len(opp)}/{len(ft)} LOT)." + ("" if len(opp) == len(ft) else " 나머지 LOT은 관계가 뚜렷하지 않다.") if len(opp) else "- 반대로 움직인 정도가 뚜렷한 LOT은 없다.",
    f"- 300분 넘게 관측된 LOT의 기울기는 온도 100분당 ±{ft.loc[ft['안정 구간[분]'].ge(300), '기울기[100분당]'].abs().max():.1f}℃, 출력 100분당 ±{fo.loc[fo['안정 구간[분]'].ge(300), '기울기[100분당]'].abs().max():.1f}%p 이하라, 시간이 지나며 서서히 바뀐 LOT은 없다."]
    + [f"- **{k}**(★)은 마지막 10%에서 LOT 중앙값보다 온도 {ft.loc[k, '마지막 10% 값']:+.1f}℃, 출력 {fo.loc[k, '마지막 10% 값']:+.1f}%p 달랐다. "
       + ("출력은 거의 그대로라, 출력 때문에 온도가 바뀐 것이 아니다." if abs(fo.loc[k, '마지막 10% 값']) < 5 else "온도와 반대 방향으로 출력이 바뀌었다(온도 변화에 제어가 따라간 모양).") for k in endbig]))
table(flow_table)


### 수준·흐름 그림을 읽는 기준

- **첫 그림 (a)(b):** 점 하나가 LOT 하나의 안정 참고 구간 중앙값, 굵은 막대가 가운데 50%다. 온도는 14 LOT이 거의 같은 값에 모여 있고, 제어출력은 강종마다 다르다.
- **둘째 그림:** LOT마다 한 칸이다. 왼쪽 축(주황)은 온도 TC-Z3, 오른쪽 축(검정)은 제어출력 OP-Z3이고, 둘 다 1분 값이다. 30분 이동 중앙값으로 그리면 몇 분짜리 오르내림이 지워져서 1분 값을 그대로 쓴다.
- **두 축 맞춤:** 두 축 모두 그 LOT의 중앙값을 가운데(점선)에 두고, 높이 1℃ = 출력 10%p로 맞췄다(1분 값에서 출력이 보통 움직이는 폭이 온도 폭의 약 9배라서, 평소 움직임이 두 선에서 비슷한 높이가 되게 했다). 그래서 두 선이 위아래 어디에 있는지는 뜻이 없고, **같은 시점에 어느 방향으로 움직였는지**만 비교한다.
- **읽는 법:** 온도가 내려갈 때 출력이 올라가면(반대로 움직임) 제어가 보충한 것이다. 출력은 그대로인데 온도만 움직이면 출력 말고 다른 이유를 찾아야 한다. 칸 제목의 "반대로 움직임"은 두 선의 순위상관이 -0.4 이하일 때다. 1분 값에서는 14 LOT 모두 반대로 움직인다.
- **마지막 10%:** LOT 기록 끝 10% 시간의 중앙값이 LOT 전체 중앙값보다 얼마나 달랐는지를 칸 제목에 적었다(그림에 따로 칠하지 않는다). 온도가 2℃ 넘게 달라진 LOT은 제목 앞에 빨간 ★를 붙였다.
- **제목 색:** 강종이다. 같은 강종은 같은 색이고, 03의 다른 그림과 같은 색을 쓴다.
- 안정 참고는 사후 형태 구분이며 정상 정답이 아니다. 22개 센서 전수 수치는 아래 표에 있다.

<a id="response-patterns"></a>

## 2. 출력이 0%·100%에 붙을 때의 온도

In [ ]:
# 출력이 0%·100%에 5분 이상 붙은 구간을, 출력이 끝에 닿은 순간(0분)에 맞춰 겹친다.
# 중앙값(굵은 선)과 가운데 50%(띠)로 보여주고, 읽을 지점에 직접 글을 단다. 회복 시간은 60분까지 본다(04와 같은 기준).
lot_ids = lots.LOT_NO.astype(int).tolist()
main_opp = opposite.loc[~opposite["예정된 변화 중"]]
sat = pd.DataFrame({f"Z{z}": [(lambda y: (y.ge(98) | y.le(2)).mean() * 100)(profile[k][(f"OP-Z{z}", "median")].where(states[k].eq("안정 참고") & quality[k][f"OP-Z{z}"].ge(.9)).dropna())
                              for k in lot_ids] for z in range(1, 9)}, index=lot_ids)
ep_rows = []
for k in lot_ids:
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9)); tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on in [("100%", op.ge(98)), ("0%", op.le(2))]:
            for a, b in runs(on.fillna(False), 5):
                if states[k].loc[a] != "안정 참고":
                    continue
                ep_rows.append({"LOT": k, "ZONE": f"Z{z}", "출력": kind, "START_MIN": a, "길이[분]": b - a,
                                "tc": (tc.reindex(range(a - 10, a + 61)) - tc.loc[a]).to_numpy(), "op": op.reindex(range(a - 10, a + 61)).to_numpy()})
saturation = pd.DataFrame(ep_rows); tt = np.arange(-10, 61)
zone_count = saturation.groupby(["ZONE", "출력"]).size().unstack(fill_value=0).reindex([f"Z{z}" for z in range(1, 9)], fill_value=0).reindex(columns=["100%", "0%"], fill_value=0)
only = zone_count.index[zone_count.sum(axis=1).gt(0)].tolist()
z3 = saturation.loc[saturation.ZONE.eq("Z3")]
fig = plt.figure(figsize=(30, 19))
gs = fig.add_gridspec(3, 2, height_ratios=[1.45, .55, 1.15], left=.06, right=.97, top=.82, bottom=.1, hspace=.42, wspace=.16)
response = {}; arrow = dict(arrowstyle="->", color="#17344d", lw=1.4)
for col, kind in enumerate(["100%", "0%"]):
    d = z3.loc[z3["출력"].eq(kind)]
    if d.empty:
        continue
    T, O = np.vstack(d.tc), np.vstack(d.op); color = "#d55e00" if kind == "100%" else "#0072b2"
    m, om = np.nanmedian(T, axis=0), np.nanmedian(O, axis=0)
    a1 = fig.add_subplot(gs[0, col]); a2 = fig.add_subplot(gs[1, col], sharex=a1)
    if len(d) <= 8:
        for t_, o_ in zip(T, O):
            a1.plot(tt, t_, color=color, lw=.9, alpha=.3); a2.plot(tt, o_, color=color, lw=.9, alpha=.3)
    else:
        a1.fill_between(tt, np.nanpercentile(T, 25, axis=0), np.nanpercentile(T, 75, axis=0), color=color, alpha=.18, lw=0)
        a2.fill_between(tt, np.nanpercentile(O, 25, axis=0), np.nanpercentile(O, 75, axis=0), color=color, alpha=.18, lw=0)
    a1.plot(tt, m, color=color, lw=3.5); a2.plot(tt, om, color=color, lw=3.5)
    for a in [a1, a2]:
        a.axvline(0, color="#17344d", lw=1.2, ls="--"); style(a, ""); a.tick_params(labelsize=12.5)
    a1.axhline(0, color="#9fb3c3", lw=1); a1.set_xlim(-10, 30); a1.set_ylim(-18, 15); a2.set_ylim(-5, 105)
    after = m[10:]
    if kind == "100%":
        i_ext = int(np.nanargmin(after[:26])); back = next((i for i in range(i_ext, len(after)) if after[i] >= -.5), None)
        response[kind] = {"사례": len(d), "더 움직인 시간[분]": i_ext, "최대 변화[℃]": after[i_ext], "원래 온도로[분]": back}
        a1.annotate(f"출력은 이미 100%인데\n{i_ext}분 더 내려가 {after[i_ext]:+.1f}℃", (i_ext, after[i_ext]), xytext=(i_ext + 5, after[i_ext] - 6), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.annotate(f"약 {back}분 뒤 원래 온도", (back, after[back]), xytext=(back + 4, 7), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.set_title(f"(a) 출력이 100%에 닿으면 — 온도는 바로 오르지 않고 {i_ext}분 더 내려간 뒤 돌아온다 ({len(d)}회 · 띠 = 가운데 50%)", loc="left", fontsize=16, pad=12)
    else:
        i_ext = int(np.nanargmax(after[:15])); j_low = int(np.nanargmin(after[i_ext:31])) + i_ext
        op_back = next((i for i in range(len(after)) if om[10 + i] >= 50), None)
        response[kind] = {"사례": len(d), "더 움직인 시간[분]": i_ext, "최대 변화[℃]": after[i_ext], "원래 온도로[분]": np.nan}
        a1.annotate(f"출력은 이미 0%인데\n{i_ext}분 더 올라 {after[i_ext]:+.1f}℃", (i_ext, after[i_ext]), xytext=(i_ext + 5, after[i_ext] + 4), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.annotate(f"출력이 돌아온 뒤({op_back}분)에도\n{j_low}분까지 {after[j_low]:+.1f}℃로 내려감", (j_low, after[j_low]), xytext=(j_low + 5, -14.5), fontsize=14, color="#17344d", arrowprops=arrow, va="center")
        a1.set_title(f"(b) 출력이 0%에 닿으면 — 온도는 {i_ext}분 더 오르고, 출력이 돌아온 뒤에는 크게 내려간다 ({len(d)}회 · 가는 선 = 한 번)", loc="left", fontsize=16, pad=12)
    a1.set_ylabel("온도 TC-Z3 변화[℃] (0분 = 0)", fontsize=13); a2.set_ylabel("제어출력 OP-Z3[%]", fontsize=13)
    a2.set_xlabel("출력이 끝에 닿은 시점 기준 시간[분]", fontsize=13); a1.tick_params(labelbottom=False)
rec_rows = []
for r in z3.loc[z3["출력"].eq("100%")].itertuples():
    after = r.tc[10:]; i_ext = int(np.nanargmin(after[:26]))
    rec_rows.append({"LOT": r.LOT, "GRADE": lot_meta.loc[r.LOT, "GRADE"], "시작": lot_meta.loc[r.LOT, "START"] + pd.Timedelta(minutes=r.START_MIN),
                     "최대 하강[℃]": after[i_ext], "원래 온도로[분]": next((i for i in range(i_ext, len(after)) if after[i] >= -.5), np.nan)})
recovery = pd.DataFrame(rec_rows)
# (c) LOT별 회복 시간: 색 대신 아래 글자로 LOT·강종·날짜를 적는다(관측 날짜 순서)
ax = fig.add_subplot(gs[2, :]); rng = np.random.default_rng(0); DOT = "#17344d"
for i, k in enumerate(lot_ids):
    r_ = recovery.loc[recovery.LOT.eq(k), "원래 온도로[분]"]
    if r_.empty:
        ax.text(i, 2, "응답\n없음", ha="center", va="bottom", fontsize=12, color="#9aa6ae")
        continue
    ax.plot(i + rng.uniform(-.12, .12, len(r_)), r_, "o", color=DOT, ms=10, alpha=.75)
    ax.plot([i - .3, i + .3], [r_.median()] * 2, color=DOT, lw=3)
    ax.text(i + .33, r_.median(), f"{r_.median():.0f}분", va="center", fontsize=11.5, color="#526d80")
med_all = recovery["원래 온도로[분]"].median()
ax.axhline(med_all, color="#9fb3c3", ls=":", lw=1.5)
rho = pd.Series([lot_ids.index(k) for k in recovery.LOT]).corr(recovery["원래 온도로[분]"].reset_index(drop=True), method="spearman")
ax.set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}\n{lot_meta.loc[k, 'START']:%m/%d}" for k in lot_ids])
ax.set_xlim(-.6, len(lot_ids) - .2); ax.set_ylim(0, recovery["원래 온도로[분]"].max() + 6)
style(ax, ""); ax.set_ylabel("원래 온도로 돌아오기까지[분]", fontsize=13); ax.tick_params(labelsize=12)
ax.set_title(f"(c) 출력을 100%로 올린 뒤 원래 온도로 돌아오기까지 걸린 시간 — Z3가 열을 더 낼 여유가 줄면 이 시간이 점점 길어진다\n"
             f"     관측 날짜 순서로 놓아도 길어지지 않는다 (순위상관 {rho:+.2f}) · 점 = 한 번, 가로선 = LOT 중앙값, 점선 = 전체 중앙값 {med_all:.0f}분",
             loc="left", fontsize=16, pad=12)
header(fig, "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나",
       f"출력이 끝에 5분 이상 붙는 일은 Z1~Z8 가운데 {', '.join(only)}에서만 {int(zone_count.loc[only].sum().sum())}회(100% {int(zone_count.loc[only, '100%'].sum())}회 · 0% {int(zone_count.loc[only, '0%'].sum())}회) · 0분 = 출력이 끝에 닿은 시점 · 1분 중앙값",
       [Patch(facecolor="#d55e00", label="출력이 100%(≥98%)에 붙은 경우"), Patch(facecolor="#0072b2", label="출력이 0%(≤2%)에 붙은 경우")], ncol=2)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, "(a)(b) 출력이 끝에 닿은 시점 기준[분] | (c) LOT 관측 날짜 순서")
finish(fig, "02_출력응답/02_출력끝값_온도반응.png")
r100 = response.get("100%", {}); r0 = response.get("0%", {})
note("\n".join([
    f"- 출력이 0% 또는 100%에 5분 이상 붙는 일은 Z1~Z8 중 {', '.join(only)}에서만 일어난다. 다른 구역의 출력은 끝까지 가지 않고 중간에서 조절된다.",
    f"- (a) Z3 출력이 100%에 닿은 뒤에도 온도는 {r100.get('더 움직인 시간[분]')}분 동안 {r100.get('최대 변화[℃]', np.nan):+.1f}℃ 더 떨어지고, 약 {r100.get('원래 온도로[분]')}분 뒤 원래 온도로 돌아온다. 출력을 최대로 올려도 온도가 바로 따라오지 않는다.",
    f"- (b) 반대로 출력이 0%에 닿은 뒤에도 온도는 {r0.get('더 움직인 시간[분]')}분 동안 {r0.get('최대 변화[℃]', np.nan):+.1f}℃ 더 오른다. 출력이 다시 올라온 뒤에도 온도는 한동안 크게 내려간다." if r0 else "- (b) 출력 0% 사례는 없다.",
    f"- (c) {len(recovery)}회 모두 60분 안에 돌아왔고 중앙값은 {med_all:.0f}분이다. 관측 날짜 순서와의 순위상관은 {rho:+.2f}로, 날짜가 지나며 길어지지 않았다. 이 응답이 없는 LOT은 '응답 없음'으로 적었다(왜 보는지는 아래 설명)."]))
table(recovery); table(sat)


### 출력·온도 그림으로 알 수 있는 것

출력이 끝(0%·100%)에 붙는 장면은 **Z3에서만** 반복된다. 출력이 100%에 닿아도 온도는 바로 오르지 않고 몇 분 더 내려간 뒤 돌아온다. 온도가 출력을 늦게 따라오는 것이다.

**(c) 원래 온도로 돌아오기까지 걸린 시간을 보는 이유**

- **무엇을 보여주나:** Z3는 온도가 내려가면 출력을 100%까지 올려서 되돌린다. 이때 돌아오는 데 걸리는 시간은 그 구역이 열을 더 낼 여유가 얼마나 남았는지를 보여준다.
- **고장과의 관계:** 히터나 버너가 약해지면 같은 100%로 올려도 회복이 점점 늦어진다. 예지보전에서 보고 싶은 것이 이런 "서서히 나빠지는 흐름"이다.
- **04와의 관계:** 04에서 Z3 반복을 "정상 반복"으로 구분하려면, 이 시간이 날짜에 따라 길어지지 않는다는 확인이 필요하다. 길어지고 있었다면 Z3 반복은 정상이 아니라 나빠지는 신호가 된다.
- **결과:** 31회 모두 60분 안에 돌아왔고 중앙값은 12분이다. 1월부터 12월까지 관측 날짜 순서로 놓아도 길어지지 않았다(순위상관 -0.14).
- **한계:** 회복 시간은 소재 두께·통판 속도 같은 부하에 따라서도 달라지는데 그 자료가 없다. 그래서 "나빠지는 흐름은 보이지 않는다"까지만 말할 수 있고, 가열 능력이 그대로라고 확정하지는 않는다.

아래 표는 같은 강종에서 6월 정기수리 전후로 Z3 출력이 끝에 붙어 있던 시간 비율이다.

In [ ]:
june = plans.iloc[0].STRT_DT
trend_rows = []
for g in ["SPHC", "SPFH590", "SPCE", "SPCEN"]:
    ids = [k for k in lot_ids if lot_meta.loc[k, "GRADE"] == g]
    pre = [k for k in ids if lot_meta.loc[k, "END"] < june]; post = [k for k in ids if lot_meta.loc[k, "START"] > june]
    trend_rows.append({"강종": g, "6월 수리 전": ", ".join(f"{k}({lot_meta.loc[k, 'START']:%m/%d}) {sat.loc[k, 'Z3']:.0f}%" for k in pre) or "-",
                       "6월 수리 후": ", ".join(f"{k}({lot_meta.loc[k, 'START']:%m/%d}) {sat.loc[k, 'Z3']:.0f}%" for k in post) or "-",
                       "변화": "줄어듦" if pre and post and max(sat.loc[post, "Z3"]) < min(sat.loc[pre, "Z3"]) else "비슷하거나 섞임"})
z3_trend = pd.DataFrame(trend_rows)
table(z3_trend)
down = z3_trend.loc[z3_trend["변화"].eq("줄어듦"), "강종"].tolist()
note(f"**결과:** Z3 출력이 0%·100%에 붙어 있던 시간은 {', '.join(down) or '어느 강종도'}에서 6월 수리 뒤 줄었지만, "
     f"{', '.join(z3_trend.loc[z3_trend['변화'].ne('줄어듦'), '강종'])}은 수리 전후가 비슷하거나 섞여 있다. 강종마다 방향이 달라 수리 효과로 단정할 수 없고, "
     "14 LOT을 날짜 순서로 놓아도 꾸준히 늘거나 주는 추세는 없다. 따라서 지금 자료로는 ①을 고장 신호로 쓰지 않고, 앞으로 LOT이 쌓일 때 같은 강종 안에서 늘어나는지 보는 감시 지표로 남긴다.")

<a id="process-levels"></a>

## 3. 과시효 Z7·Z8와 세정 입구의 수준 차이

In [ ]:
# 같은 강종 다른 LOT 중앙값(–)에서 이 LOT 값(●)까지 화살표로 잇는다. 볼 LOT만 빨강, 나머지는 회색.
lg = level_gap.set_index(["LOT", "SENSOR"])
fig, axs = plt.subplots(2, 1, figsize=(30, 15), sharex=True)
fig.subplots_adjust(left=.05, right=.98, top=.8, bottom=.09, hspace=.32)
for ax, kind in zip(axs, ["TC", "OP"]):
    for s, dx, mk in [(f"{kind}-Z7", -.16, "o"), (f"{kind}-Z8", .16, "s")]:
        for i, k in enumerate(lot_ids):
            z = lg.loc[(k, s)]; on = k in OVERAGE_LOTS; c = SELECT_COLOR if on else "#9aa6ae"; xx = i + dx
            if pd.notna(z.PEER_MEDIAN):
                ax.plot(xx, z.PEER_MEDIAN, marker="_", color="#17344d" if on else "#7c8995", ms=22, mew=2.5)
                ax.annotate("", (xx, z.LOT_MEDIAN), (xx, z.PEER_MEDIAN), arrowprops=dict(arrowstyle="-|>", color=c, lw=3 if on else 1.4, mutation_scale=16))
            ax.plot(xx, z.LOT_MEDIAN, mk, color=c, ms=13 if on else 10, zorder=4)
    for k in OVERAGE_LOTS:
        z = lg.loc[(k, f"{kind}-Z7")]
        ax.text(lot_ids.index(k), max(z.LOT_MEDIAN, z.PEER_MEDIAN) + (.25 if kind == "TC" else .6), f"{z.GAP:+.1f}{'℃' if kind == 'TC' else '%p'}",
                ha="center", va="bottom", fontsize=14, fontweight="bold", color=SELECT_COLOR)
    style(ax, ""); ax.set_ylabel("온도[℃]" if kind == "TC" else "출력[%]", fontsize=13); ax.tick_params(labelsize=12.5)
ov = level_gap.loc[level_gap.LOT.isin(OVERAGE_LOTS)].set_index(["LOT", "SENSOR"]).GAP
op_gaps = [ov[(k, s)] for k in OVERAGE_LOTS for s in ["OP-Z7", "OP-Z8"]]
axs[0].set_title(f"(a) 과시효 온도 [Z7 ○ · Z8 □]: {', '.join(map(str, OVERAGE_LOTS))}만 같은 강종보다 약 +3℃ 높다", loc="left", fontsize=17, pad=12)
axs[1].set_title(f"(b) 같은 구역 출력 [Z7 ○ · Z8 □]: 이 세 LOT의 출력은 같은 강종보다 {min(op_gaps):+.1f}~{max(op_gaps):.1f}%p — 출력은 그대로인데 온도만 높다", loc="left", fontsize=17, pad=12)
axs[1].set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}" for k in lot_ids])
for t, k in zip(axs[1].get_xticklabels(), lot_ids):
    t.set_color(SELECT_COLOR if k in OVERAGE_LOTS else GRADE_COLOR[lot_meta.loc[k, "GRADE"]]); t.set_fontweight("bold" if k in OVERAGE_LOTS else "normal")
header(fig, "과시효 Z7·Z8 수준 · LOT별", "화살표 = 같은 강종 다른 LOT 중앙값(–)에서 이 LOT 값(●)까지 · 안정 참고 1분 중앙값 · SGCC·DP590·SPCD는 같은 강종 다른 LOT이 없어 점만 표시",
       [Line2D([0], [0], marker="_", color="#17344d", ls="None", ms=20, mew=2.5, label="– = 같은 강종 다른 LOT 중앙값 (출발)"),
        Line2D([0], [0], marker="o", color=SELECT_COLOR, ls="-", lw=3, ms=12, label="빨강 = 같은 강종보다 Z7·Z8이 함께 높은 LOT (도착)"),
        Line2D([0], [0], marker="o", color="#9aa6ae", ls="-", lw=1.4, ms=10, label="회색 = 나머지 LOT"),
        Line2D([0], [0], marker="o", color="#455a64", ls="None", ms=10, label="○ Z7"), Line2D([0], [0], marker="s", color="#455a64", ls="None", ms=10, label="□ Z8")], ncol=5)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, compare_tag("LOT별 과시효 수준"))
finish(fig, "03_과시효_세정/03_과시효_Z7Z8.png")
table(level_gap.loc[level_gap.LOT.isin(OVERAGE_LOTS) & level_gap.SENSOR.isin(["TC-Z6", "TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8"])].pivot(index="LOT", columns="SENSOR", values="GAP").round(2))
rest = level_gap.loc[~level_gap.LOT.isin(OVERAGE_LOTS) & level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.notna() & level_gap.GAP.lt(2.5)]
note("\n".join([
    f"- (a) 빨간 LOT {', '.join(f'{k}({lot_meta.loc[k, 'GRADE']})' for k in OVERAGE_LOTS)}은 과시효 Z7·Z8이 함께 같은 강종보다 {min(ov[(k, s)] for k in OVERAGE_LOTS for s in ['TC-Z7', 'TC-Z8']):+.1f}~{max(ov[(k, s)] for k in OVERAGE_LOTS for s in ['TC-Z7', 'TC-Z8']):+.1f}℃ 높다. 회색 LOT의 차이는 {rest.GAP.min():+.1f}~{rest.GAP.max():+.1f}℃다.",
    f"- (b) 같은 구역 출력은 이 세 LOT에서 같은 강종보다 {min(op_gaps):+.1f}~{max(op_gaps):.1f}%p로 오히려 같거나 낮다. 온도가 3℃ 높은데 출력은 더 쓰지 않았다.",
    "- 그래서 과시효 설정 온도가 LOT마다 달랐는지, Z7·Z8 온도 측정값이 함께 치우쳤는지를 확인해야 한다. 제공된 부품교체 기록에는 온도 센서 교체가 없다."]))


In [ ]:
# 위: LOT 중앙값을 날짜 순서로 · 아래: 같은 강종(SPCE) LOT의 1분 중앙값 곡선
hi_mean, lo_mean = cln[cln >= CLN_SPLIT].mean(), cln[cln < CLN_SPLIT].mean()
HI_C, LO_C = "#16a085", "#5d6d7e"
fig = plt.figure(figsize=(30, 17))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.05], left=.05, right=.97, top=.83, bottom=.06, hspace=.45)
ax = fig.add_subplot(gs[0])
ax.plot(range(len(lot_ids)), [cln[k] for k in lot_ids], color="#b9c4cc", lw=1.5, zorder=1)
for i, k in enumerate(lot_ids):
    hi = cln[k] >= CLN_SPLIT
    ax.plot(i, cln[k], "o", ms=16, color=HI_C if hi else LO_C, zorder=3)
    ax.text(i, cln[k] + (.35 if hi else -.35), f"{cln[k]:.1f}", ha="center", va="bottom" if hi else "top", fontsize=13, color="#17344d")
for m_, label, c in [(hi_mean, "높은 수준", HI_C), (lo_mean, "낮은 수준", LO_C)]:
    ax.axhline(m_, color=c, ls="--", lw=1.3)
    ax.text(len(lot_ids) - .4, m_ + (.15 if c == HI_C else -.15), f" {label} 평균 {m_:.1f}℃", va="bottom" if c == HI_C else "top", fontsize=13, color=c, fontweight="bold")
switch_rows = []
for i in range(len(lot_ids) - 1):
    a, b = lots.iloc[i], lots.iloc[i + 1]
    n = int(cal_work.STRT_DT.between(a.END, b.START).sum()); sw = (cln[lot_ids[i]] >= CLN_SPLIT) != (cln[lot_ids[i + 1]] >= CLN_SPLIT)
    switch_rows.append({"앞 LOT": lot_ids[i], "뒤 LOT": lot_ids[i + 1], "그 사이 정비[WO]": n, "수준 바뀜": sw})
    ax.text(i + .5, 63.3, f"정비 {n}", ha="center", fontsize=12, color=SELECT_COLOR if (sw and n == 0) else "#7c8995", fontweight="bold" if (sw and n == 0) else "normal")
switches = pd.DataFrame(switch_rows); zero_switch = int((switches["수준 바뀜"] & switches["그 사이 정비[WO]"].eq(0)).sum())
ax.set_ylim(62.8, 71.5); ax.set_xlim(-.5, len(lot_ids) + .9)
ax.set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}" for k in lot_ids])
for t, k in zip(ax.get_xticklabels(), lot_ids):
    t.set_color(GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
style(ax, ""); ax.set_ylabel("CLN-IN LOT 중앙값[℃]", fontsize=13); ax.tick_params(labelsize=12.5)
ax.set_title(f"(a) 세정 입구 온도는 LOT마다 약 {hi_mean:.1f}℃ 또는 {lo_mean:.1f}℃ 두 수준 중 하나 — 정비가 0건인 LOT 사이에서도 {zero_switch}번 바뀐다 (빨간 '정비 0')", loc="left", fontsize=16, pad=12)
bx = fig.add_subplot(gs[1])
spce = [k for k in lot_ids if lot_meta.loc[k, "GRADE"] == "SPCE"]
for k, c in zip(spce, ["#5d6d7e", "#9aa6ae", "#7f8c8d"]):
    y = profile[k][("CLN-IN", "median")]; hi = cln[k] >= CLN_SPLIT
    bx.plot(y.index, y, lw=2.4 if hi else 1.8, color=HI_C if hi else c, label=f"{k} · {lot_meta.loc[k, 'START']:%m/%d} · {'높은' if hi else '낮은'} 수준 {cln[k]:.1f}℃")
bx.set_xlim(0, lots.loc[lots.GRADE.eq("SPCE"), "MINUTES"].max() + 5); style(bx, "관측 시작 후 경과시간[분]"); bx.set_ylabel("CLN-IN[℃] (1분 중앙값)", fontsize=13)
bx.legend(fontsize=13, frameon=False, loc="upper right"); bx.tick_params(labelsize=12.5)
low_spce = [k for k in spce if cln[k] < CLN_SPLIT]; high_spce = [k for k in spce if cln[k] >= CLN_SPLIT]
bx.set_title(f"(b) 같은 SPCE: 낮은 LOT({'·'.join(map(str, low_spce))})은 천천히 올랐다가 한 번에 떨어지는 반복(톱니)이 이어지고, 높은 LOT({'·'.join(map(str, high_spce))})은 처음부터 약 3℃ 위에 있다",
             loc="left", fontsize=16, pad=12)
header(fig, "세정 입구 온도 (CLN-IN)", "위: 점 = LOT 중앙값(날짜 순서), 점 사이 숫자 = 두 LOT 사이에 시작한 CAL01 정비 수 · 아래: 같은 강종 LOT의 1분 중앙값",
       [Line2D([0], [0], marker="o", color=HI_C, ls="None", ms=12, label=f"높은 수준 LOT ({int((cln >= CLN_SPLIT).sum())}개)"),
        Line2D([0], [0], marker="o", color=LO_C, ls="None", ms=12, label=f"낮은 수준 LOT ({int((cln < CLN_SPLIT).sum())}개)")], ncol=2)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, "(a) LOT 날짜 순서 | (b) " + TAG_ELAPSED)
finish(fig, "03_과시효_세정/04_세정입구.png")
table(switches)


### 세정 입구 온도로 알 수 있는 것

- **두 수준:** 14 LOT의 세정 입구 온도 중앙값은 약 69.5℃ 묶음과 약 66.3℃ 묶음으로 나뉜다. 같은 강종 안에서도 갈리므로 강종으로 정해지는 값이 아니다.
- **정비 이력과의 관계:** 위 표처럼 두 LOT 사이에 정비가 0건이어도 수준이 바뀐 경우가 있어, 정비로 설명되지 않는다. 정비 이력과 함께 볼 필요는 없다.
- **LOT 안의 변동:** 천천히 오르다 한 번에 1~2℃ 떨어지는 톱니 모양이 두 수준 모두에서 반복된다. 세정 구역 자체의 운전 반복으로 보이며, 소둔로 상태를 알려주는 신호가 아니다.
- **그래서 어떻게 쓰나:** 세정 입구 온도는 LOT끼리 비교해 이상을 찾는 값이 아니라 **운전 조건**으로 다룬다. 같은 강종 대비 차이에서 세정 입구가 다르게 나오는 LOT은 이 수준 차이 때문이다.
- 세정 설정값·운전 기록이 없어 두 수준이 생긴 이유는 확인하지 않는다.

<a id="cp-patterns"></a>

## 4. CP 원본과 동시 신호

In [ ]:
_below = []
cps = cp_candidates.reset_index(drop=True)
fig, axs = plt.subplots(len(cps), 2, figsize=(26, 4 + 3.25 * len(cps)))
fig.subplots_adjust(left=.08, right=.975, top=.87, bottom=0.04, wspace=.18, hspace=1.05)
cp_rows = []
for i, c in cps.iterrows():
    k = int(c.LOT); r = lot_meta.loc[k]
    a, b_ = max(0, c.START_MIN - 5), min(r.MINUTES, c.END_MIN + 5)
    w = raw_window(k, a, b_)
    for j, s in enumerate(["CP-Z4", "CP-Z4M"]):
        ax = axs[i, j]
        scale = 1e10 if s == "CP-Z4M" else 1           # 지수 표기가 위쪽 보조축과 겹치지 않도록 축 이름에 배율을 적는다
        ax.plot(w.index, w[s] * scale, color=LOT_COLOR[k], lw=1.3)
        ax.axvspan(r.START + pd.Timedelta(minutes=c.START_MIN), r.START + pd.Timedelta(minutes=c.END_MIN), color=COL["FOCUS"], alpha=.1, lw=0)
        if s == "CP-Z4":
            ax.axhline(c.BASE, color="#455a64", ls="--", lw=1)
        ax.set_xlim(w.index[0], w.index[-1])
        ax.set_ylabel("CP-Z4M 기록값[×1e-10]" if scale > 1 else unit(s), labelpad=8, fontsize=10.5)
        clock_axis(ax, w.index[0])
        elapsed_top(ax, r.START)
        ax.yaxis.set_major_locator(MaxNLocator(4))
        title = f"{k} · {r.GRADE} · [{s}]" + (f" · 기준 대비 {c.DELTA:+.3f}" if s == "CP-Z4" else "")
        ax.set_title(title, loc="left", fontsize=13.5, pad=36, color=LOT_COLOR[k])
    at_peak = cal_work.loc[cal_work.STRT_DT.le(c.PEAK_TIME) & cal_work.END_DT.ge(c.PEAK_TIME)]
    cp_rows.append({"LOT": k, "GRADE": r.GRADE, "극값 시각": c.PEAK_TIME, "극값": c.PEAK_VALUE, "기준 대비": c.DELTA,
                    "펄스 폭[분]": c.PULSE_WIDTH_MIN, "초과 기록[행]": c.RECORDED_SECONDS, "극값 시각의 CAL01 작업[WO]": len(at_peak)})
cp_context = pd.DataFrame(cp_rows)
header(fig, "CP 원본 7 LOT", "후보 펄스 앞뒤 5분의 1초 원본; 연분홍=기준 대비 0.05 이상 벗어난 펄스, 점선=LOT 자체 안정 참고 중앙값",
       [Patch(facecolor=COL["FOCUS"], alpha=.2, label="CP 후보 펄스"), Line2D([0], [0], color="#455a64", ls="--", label="자체 안정 참고 중앙값")], ncol=2)
compact_header(fig, .955, .937)
axis_tag(fig, TAG_REAL)
_below.append(f"7 LOT 모두 원본에서 기준 대비 0.05 이상 벗어난 기록이 {cps.RECORDED_SECONDS.min()}–{cps.RECORDED_SECONDS.max()}행 있다 · 극값 시각에 CAL01 작업이 있던 LOT은 {int(cp_context['극값 시각의 CAL01 작업[WO]'].gt(0).sum())}개")
finish(fig, "04_CP/05_CP_원본7LOT.png")
note("\n".join("- " + t.strip() for t in _below))
table(cp_context)
verdict("CP 차이 ≥0.05는 원본 여러 기록에서 관측된다", "지지",
        f"{len(cps)} LOT의 극값 포함 펄스 초과 기록 {cps.RECORDED_SECONDS.min()}–{cps.RECORDED_SECONDS.max()}행",
        "원본 피크·펄스·전체 신호와 표본 수를 전수 확인", "CP 검토는 해당 분·센서에만 표시")

In [ ]:
k = int(CP_TOP.LOT); r = lot_meta.loc[k]; g = raw[k].set_index("MEAS_DT")
a0 = r.START + pd.Timedelta(minutes=CP_TOP.START_MIN); a1 = r.START + pd.Timedelta(minutes=CP_TOP.END_MIN)
pre = g.loc[a0 - pd.Timedelta(minutes=15):a0 - pd.Timedelta(seconds=1)]; dur = g.loc[a0:a1]
shift_rows = []
for s in SIGNALS:
    iqr = pre[s].quantile(.75) - pre[s].quantile(.25)
    shift_rows.append({"센서": s, "앞 15분 중앙값": pre[s].median(), "펄스 중 중앙값": dur[s].median(), "앞 15분 IQR": iqr,
                       "IQR 대비 이동[배]": (dur[s].median() - pre[s].median()) / iqr if iqr > 0 else np.nan})
shift = pd.DataFrame(shift_rows).set_index("센서")
cor_rows = []
for kk in lot_ids:
    p = profile[kk]; m = states[kk].eq("안정 참고")
    x = p[("CP-Z4", "median")].where(m).diff()
    for s in SIGNALS:
        if s != "CP-Z4":
            cor_rows.append({"LOT": kk, "센서": s, "r": x.corr(p[(s, "median")].where(m).diff())})
cors = pd.DataFrame(cor_rows).groupby("센서").r.agg(["median", "min", "max"]).reindex([s for s in SIGNALS if s != "CP-Z4"])

fig, (ax, bx) = plt.subplots(2, 1, figsize=(30, 15))
fig.subplots_adjust(left=.06, right=.98, top=.80, bottom=.1, hspace=.55)
others = shift.drop(index="CP-Z4")
colors = ["#0072b2" if s in TC else "#009e73" if s in OP else "#7c8995" for s in others.index]
ax.bar(range(len(others)), others["IQR 대비 이동[배]"], color=colors, width=.65)
for i, v in enumerate(others["IQR 대비 이동[배]"]):
    if pd.notna(v):
        ax.text(i, v, f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=12)
ax.axhline(0, color="#9fb3c3", lw=1)
ax.set_xticks(range(len(others)), others.index, rotation=0)
ax.set_title(f"(a) {k} CP 펄스({CP_TOP.PEAK_TIME:%m/%d %H:%M}, {CP_TOP.PULSE_WIDTH_MIN:.1f}분) 동안 각 센서의 이동 · CP-Z4 자신은 {shift.loc['CP-Z4', 'IQR 대비 이동[배]']:+.0f}배",
             loc="left", fontsize=16, pad=12)
ax.set_ylabel("앞 15분 IQR 대비[배]", labelpad=8); style(ax, ""); ax.tick_params(labelsize=12)
bx.bar(range(len(cors)), cors["median"], color=["#0072b2" if s in TC else "#009e73" if s in OP else "#7c8995" for s in cors.index], width=.65)
bx.vlines(range(len(cors)), cors["min"], cors["max"], color="#455a64", lw=1.4)
for lim in [-.3, .3]:
    bx.axhline(lim, color="#526d80", ls=":", lw=1.2)
bx.axhline(0, color="#9fb3c3", lw=1)
bx.set_ylim(-1, 1)
bx.set_xticks(range(len(cors)), cors.index)
bx.set_title("(b) 14 LOT · CP-Z4 1분 변화량과 각 센서 1분 변화량의 상관 · 막대 = 중앙값, 선 = 최소~최대, 점선 = ±0.3", loc="left", fontsize=16, pad=12)
bx.set_ylabel("상관계수 r", labelpad=8); style(bx, ""); bx.tick_params(labelsize=12)
header(fig, "CP와 다른 센서의 관계", "CP가 움직일 때 온도·출력도 함께 움직이는지 · 파랑 = 온도, 초록 = 출력, 회색 = CP-Z4M",
       [Patch(facecolor="#0072b2", label="온도 TC"), Patch(facecolor="#009e73", label="출력 OP"), Patch(facecolor="#7c8995", label="CP-Z4M")], ncol=3)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, compare_tag("CP와 다른 센서"))
finish(fig, "04_CP/07_CP_다른센서_관계.png")
table(shift); table(cors.round(2))
top = others["IQR 대비 이동[배]"].abs().sort_values(ascending=False).index[:3]
note("\n".join([
    f"- (a) 펄스 동안 CP-Z4는 평소 흔들림의 {shift.loc['CP-Z4', 'IQR 대비 이동[배]']:+.0f}배 움직였지만, 다른 센서는 가장 큰 것도 " +
    ", ".join(f"{s} {others.loc[s, 'IQR 대비 이동[배]']:+.1f}배({others.loc[s, '펄스 중 중앙값'] - others.loc[s, '앞 15분 중앙값']:+.1f})" for s in top) + "다. 신호별 실제 차이와 기존 변동폭을 함께 읽으며, 냉각 신호는 반복 파형의 위상도 고려한다.",
    f"- (b) 14 LOT에서 CP-Z4의 1분 변화와 다른 21개 센서의 1분 변화의 상관은 중앙값 기준 {cors['median'].min():+.2f}~{cors['median'].max():+.2f}다. 이는 CP-Z4의 전체 요약이며 구간별 동반 변화나 CP-Z4M의 관계가 없다는 뜻은 아니다.",
    "- CP-Z4의 전체 상관 요약만으로 국소 동반 변화를 배제할 수 없다. CP-Z4와 CP-Z4M을 나누어 구간별 온도·출력과 대조한다. 두 값은 무차원이며, 산식과 기록 의미를 확인해야 영향 방향을 해석할 수 있다."]))

### CP-Z4와 CP-Z4M을 나누어 읽기

CP-Z4만을 기준으로 다른 센서와 상관을 계산하면 CP-Z4M 자체의 온도 연관성을 놓칠 수 있다. 아래는 **14 LOT 모두**에서 안정 참고·센서별 기록량 ≥90%인 1분 중앙값을 사용한 대조다. 수준과 1분 변화량을 함께 확인한다. CP-Z4M의 아주 작은 값을 0으로 반올림하지 않는다.

구간표의 온도·출력 차이는 **CP-Z4 검토 구간 중앙값 − 직전 15분 중앙값**이다. CP 극값의 기준 대비 차이는 자체 안정 참고 중앙값을 사용하므로 두 비교 기준을 구분한다. 구간 길이가 서로 다르고 제어 동작·운전 조건이 섞여 있어 전후 차이를 인과 효과로 해석하지 않는다.


In [ ]:
cp_channel_rows=[]
for cp_lot in lots.LOT_NO.astype(int):
    cp_profile=profile[cp_lot]
    cp_stable=states[cp_lot].eq("안정 참고")
    for cp_channel in ["CP-Z4", "CP-Z4M"]:
        cp_x=cp_profile[(cp_channel,"median")].where(cp_stable & quality[cp_lot][cp_channel].ge(.9))
        for cp_sensor in ["TC-Z5", "OP-Z5"]:
            cp_y=cp_profile[(cp_sensor,"median")].where(cp_stable & quality[cp_lot][cp_sensor].ge(.9))
            cp_pair=pd.concat([cp_x.rename("CP"),cp_y.rename("SENSOR")],axis=1).dropna()
            cp_diff=pd.concat([cp_x.diff().rename("CP"),cp_y.diff().rename("SENSOR")],axis=1).dropna()
            cp_channel_rows.append({"LOT":cp_lot,"CP 항목":cp_channel,"대조 신호":cp_sensor,
                                    "수준 유효 분":len(cp_pair),"수준 상관":cp_pair.CP.corr(cp_pair.SENSOR),
                                    "변화량 유효 분":len(cp_diff),"1분 변화량 상관":cp_diff.CP.corr(cp_diff.SENSOR)})
cp_channel_relations=pd.DataFrame(cp_channel_rows)
cp_m_tc5=cp_channel_relations.loc[cp_channel_relations["CP 항목"].eq("CP-Z4M") & cp_channel_relations["대조 신호"].eq("TC-Z5")]
table(cp_m_tc5)
cp_m_op5=cp_channel_relations.loc[cp_channel_relations["CP 항목"].eq("CP-Z4M") & cp_channel_relations["대조 신호"].eq("OP-Z5")]
note(f"**확인된 동반 변화:** CP-Z4M–TC-Z5의 1분 변화량 상관은 14 LOT에서 {cp_m_tc5['1분 변화량 상관'].min():.3f}–{cp_m_tc5['1분 변화량 상관'].max():.3f}, 중앙값 {cp_m_tc5['1분 변화량 상관'].median():.3f}다. CP-Z4M–OP-Z5는 {cp_m_op5['1분 변화량 상관'].min():.3f}–{cp_m_op5['1분 변화량 상관'].max():.3f}다. 온도와 CP-Z4M은 같은 방향, 출력은 대체로 반대 방향으로 움직인다.")

cp_interval_rows=[]
for cp_event in cp_candidates.itertuples():
    cp_lot=int(cp_event.LOT)
    cp_g=raw[cp_lot].set_index("MEAS_DT")
    cp_start=lot_meta.loc[cp_lot,"START"]+pd.Timedelta(seconds=round(cp_event.START_MIN*60))
    cp_end=lot_meta.loc[cp_lot,"START"]+pd.Timedelta(seconds=round(cp_event.END_MIN*60))
    cp_pre=cp_g.loc[(cp_g.index>=cp_start-pd.Timedelta(minutes=15)) & (cp_g.index<cp_start)]
    cp_during=cp_g.loc[(cp_g.index>=cp_start) & (cp_g.index<cp_end)]
    cp_result={"LOT":cp_lot,"강종":cp_event.GRADE,"CP 구간 시작":cp_start,"CP 구간 끝(제외)":cp_end,
               "CP 극값":cp_event.PEAK_VALUE,"자체 참고 대비":cp_event.DELTA,"극값 시각":cp_event.PEAK_TIME}
    for cp_sensor in ["TC-Z3","OP-Z3","TC-Z4","OP-Z4","TC-Z5","OP-Z5","CP-Z4M"]:
        cp_result[cp_sensor+" 중앙값 차이"]=cp_during[cp_sensor].median()-cp_pre[cp_sensor].median()
    cp_interval_rows.append(cp_result)
cp_interval_relations=pd.DataFrame(cp_interval_rows)
table(cp_interval_relations)
note("**우선 확인할 구간:** 240069의 07/08 02:30–02:39, 240117의 11/13 13:03–13:06, 240024의 03/08 04:59–05:00, 240058의 06/09 종료 전환이다. CP-Z4 극값 시각에는 CAL01 실제 작업이 없으며, 240117은 CBM 완료 뒤의 구간이다. 정비와의 시간 관계는 02의 해당 LOT 원본에서 확인한다.")
note("**영향 방향의 한계:** 강한 동시 관계는 온도가 CP를 바꿨거나 CP가 온도를 바꿨다는 증명이 아니다. CP-Z4M의 산식·생성 규칙에 TC-Z5가 포함되는지 먼저 확인해야 한다. 산출값이면 강한 상관은 계산 구조에서 생길 수 있다. OP-Z5와의 반대 움직임 역시 온도 제어 반응과 다른 공통 요인을 구분해야 한다.")


### CP로 알 수 있는 것

자체 참고 수준에서 벗어난 방향·크기·지속시간과 다른 센서의 동시 변화다. ±0.05는 탐색 문턱이며 정상 한계가 아니다. 같은 SPHC에서도 변화 모양과 크기가 달라 강종 고유 특성으로 단정할 수 없다. 전수 7 LOT을 보존하며 각 LOT 실제 이력과 원본은 02에 연결한다. 상관·IQR 대비 이동은 연관성 단서이며 원인이나 고장 진단이 아니다.

<a id="relationships"></a>

## 5. 온도·출력 연관성 — 확인용 표

In [ ]:
# 같은 구역 온도(TC)와 출력(OP)의 상관: 제어가 정상적으로 작동하는지 확인하는 표로만 남긴다.
corr_rows = []
for r in lots.itertuples():
    k = int(r.LOT_NO); p = profile[k]
    for z in range(1, 9):
        tc, op = f"TC-Z{z}", f"OP-Z{z}"
        valid = states[k].eq("안정 참고") & quality[k][tc].ge(.9) & quality[k][op].ge(.9)
        pair = pd.DataFrame({"TC": p[(tc, "median")].where(valid), "OP": p[(op, "median")].where(valid)}).dropna()
        d = pair.diff().dropna()
        corr_rows.append({"LOT": k, "GRADE": r.GRADE, "ZONE": f"Z{z}",
                          "수준 상관": pair.corr(method="spearman").iloc[0, 1] if len(pair) >= 30 else np.nan,
                          "1분 변화 상관": d.corr(method="spearman").iloc[0, 1] if len(d) >= 30 else np.nan})
corr_table = pd.DataFrame(corr_rows)
corr_zone = corr_table.groupby("ZONE")[["수준 상관", "1분 변화 상관"]].agg(["min", "median", "max"]).round(2)
table(corr_zone)
note(f"**결과:** 14 LOT × 8구역 모두에서 온도와 출력은 반대로 움직인다(수준 상관 {corr_table['수준 상관'].min():+.2f}~{corr_table['수준 상관'].max():+.2f}, "
     f"1분 변화 상관 {corr_table['1분 변화 상관'].min():+.2f}~{corr_table['1분 변화 상관'].max():+.2f}). 온도가 오르면 출력을 줄이는 제어가 모든 LOT·구역에서 정상적으로 작동한다는 확인이며, "
     "다르게 움직이는 LOT이 없어 따로 볼 구간은 없다. 그래서 그림 없이 표로만 남긴다.")
table(corr_table)


<a id="all-raw-evidence"></a>

## 6. 원본 후보의 전수 확인

In [ ]:
raw_delta,temperature_rows={},[]
def seconds_series(k,s):
    g=raw[k];ix=np.rint(g.ELAPSED.to_numpy()*60).astype(int)
    grid=pd.RangeIndex(int(np.floor(lot_meta.loc[k,"MINUTES"]*60))+1)
    y=pd.Series(g[s].to_numpy(),index=ix).reindex(grid)
    minute=grid.to_numpy()//60
    valid=states[k].reindex(minute).to_numpy()=="안정 참고"
    valid &= quality[k][s].reindex(minute).to_numpy()>=.9
    return y.where(valid)
for e in episodes.itertuples():
    key=e.LOT,e.SENSOR
    if key not in raw_delta:
        ids=lots.loc[lots.GRADE.eq(e.GRADE),"LOT_NO"].astype(int).tolist()
        target=seconds_series(e.LOT,e.SENSOR)
        peers=pd.DataFrame({k:seconds_series(k,e.SENSOR).reindex(target.index) for k in ids if k!=e.LOT})
        n=peers.count(axis=1)
        d=(target-peers.median(axis=1)).where(n.ge(2))
        raw_delta[key]=pd.DataFrame({"DELTA":d,"N_PEERS":n})
    block=raw_delta[key].loc[int(e.START_MIN*60):int(e.END_MIN*60)-1]
    v=block.DELTA.dropna();hit=block.DELTA.abs().ge(3)
    run=runs(hit);maxrun=max([b-a for a,b in run],default=0)
    ratio=hit.sum()/len(v) if len(v) else np.nan
    temperature_rows.append({"EPISODE_ID":e.Index+1,"LOT":e.LOT,"GRADE":e.GRADE,"SENSOR":e.SENSOR,
                             "START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"DURATION_MIN":e.DURATION_MIN,
                             "RAW_COMPARE_SECONDS":len(v),"RAW_EXCEED_SECONDS":int(hit.sum()),"RAW_EXCEED_RATIO":ratio,
                             "MAX_CONTIGUOUS_RECORDED_S":maxrun,"RAW_MAX_ABS_DELTA_C":v.abs().max(),
                             "VERDICT":"원본 반복 확인" if len(v)>=60 and ratio>=.5 else "비율 부족·보류"})
temperature_validation=pd.DataFrame(temperature_rows)
window_rows=[]
for k,sub in episodes.groupby("LOT"):
    current=None
    for e in sub.sort_values(["START_MIN","END_MIN"]).itertuples():
        if current is None:
            current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
        elif e.START_MIN<=current["END_MIN"]+5:
            current["END_MIN"]=max(current["END_MIN"],e.END_MIN);current["SENSORS"].add(e.SENSOR)
        else:
            window_rows.append(current);current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
    if current is not None:window_rows.append(current)
temperature_windows=pd.DataFrame(window_rows)
note(f"**전수 결과:** {len(temperature_validation)}개 센서 후보 중 {(temperature_validation.VERDICT=='원본 반복 확인').sum()}개는 원본 비교 초의 절반 이상에서 3℃ 차이가 반복된다. 보류는 {temperature_validation.VERDICT.ne('원본 반복 확인').sum()}개다. 총 {len(temperature_windows)}개의 확대창에서 원본·해당 OP·개별 후보를 아래에 모두 보여준다.")

## 04로 이어서 확인

[04 탐지·구간 근거](05_all_data_timeseries_jinny_04.ipynb)에서 사람에게 보이는 차이와 탐지 방법의 후보 위치를 대조한다. 정답 라벨이 없어 탐지를 고장 확정이나 예측 성능으로 해석하지 않는다.

In [ ]:
table(pd.DataFrame(figure_checks))